In [1]:
from __future__ import annotations

from collections import Counter
from typing import Dict, List, Tuple

Token = str
Pair = Tuple[Token, Token]
WordTokens = Tuple[Token, ...]



def build_initial_vocab(words_with_freq: Dict[str, int]) -> Dict[WordTokens, int]:
    """Represent each word as characters + '_' end marker."""
    return {tuple(list(w) + ["_"]): f for w, f in words_with_freq.items()}


def get_pair_counts(vocab: Dict[WordTokens, int]) -> Counter[Pair]:
    """Count adjacent token pairs in the current vocabulary (weighted by word frequency)."""
    pairs = Counter()
    for toks, freq in vocab.items():
        for i in range(len(toks) - 1):
            pairs[(toks[i], toks[i + 1])] += freq
    return pairs


def merge_pair_in_vocab(vocab: Dict[WordTokens, int], pair: Pair, new_token: Token) -> Dict[WordTokens, int]:
    """Replace occurrences of `pair` with `new_token` in every word token sequence."""
    a, b = pair
    new_vocab: Dict[WordTokens, int] = {}

    for toks, freq in vocab.items():
        i = 0
        out: List[Token] = []
        while i < len(toks):
            if i < len(toks) - 1 and toks[i] == a and toks[i + 1] == b:
                out.append(new_token)
                i += 2
            else:
                out.append(toks[i])
                i += 1

        out_t = tuple(out)
        new_vocab[out_t] = new_vocab.get(out_t, 0) + freq

    return new_vocab


def vocabulary_size(vocab: Dict[WordTokens, int]) -> int:
    """Number of unique tokens currently used."""
    uniq = set()
    for toks in vocab:
        uniq.update(toks)
    return len(uniq)


def choose_best_pair(pair_counts: Counter[Pair]) -> Tuple[Pair, int]:
    """
    Choose the most frequent pair.
    Tie-break rule: among max-count pairs, pick lexicographically smallest pair.
    """
    if not pair_counts:
        raise ValueError("No pairs to choose from.")
    max_count = max(pair_counts.values())
    best_pairs = [p for p, c in pair_counts.items() if c == max_count]
    best_pair = min(best_pairs)
    return best_pair, max_count


def learn_bpe_merges(words_with_freq: Dict[str, int], num_merges: int = 30):
    """
    Learn BPE merges for given training data.
    Prints best pair and evolving vocab size at each step.
    Returns list of merges as tuples: ((a,b), new_token, count_at_merge_time)
    """
    vocab = build_initial_vocab(words_with_freq)
    merges: List[Tuple[Pair, Token, int]] = []

    print("=== BPE Training ===")
    print(f"Initial vocab size (unique tokens): {vocabulary_size(vocab)}\n")

    for step in range(1, num_merges + 1):
        pair_counts = get_pair_counts(vocab)
        if not pair_counts:
            print("No more pairs to merge.")
            break

        best_pair, best_count = choose_best_pair(pair_counts)
        new_tok = "".join(best_pair)

        vocab = merge_pair_in_vocab(vocab, best_pair, new_tok)
        merges.append((best_pair, new_tok, best_count))

        print(
            f"Step {step:02d}: best_pair={best_pair} count={best_count} "
            f"-> new_token='{new_tok}' | vocab_size={vocabulary_size(vocab)}"
        )

    return merges


def apply_bpe(word: str, merges: List[Tuple[Pair, Token, int]]) -> List[Token]:
    """Segment a word by applying learned merges in order."""
    toks: List[Token] = list(word) + ["_"]

    for (a, b), new_tok, _cnt in merges:
        i = 0
        out: List[Token] = []
        while i < len(toks):
            if i < len(toks) - 1 and toks[i] == a and toks[i + 1] == b:
                out.append(new_tok)
                i += 2
            else:
                out.append(toks[i])
                i += 1
        toks = out

    return toks



def toy_corpus_freqs() -> Dict[str, int]:
    """
    Toy corpus from class:
    low x5, lowest x2, newer x6, wider x3, new x2
    """
    words = ["low"] * 5 + ["lowest"] * 2 + ["newer"] * 6 + ["wider"] * 3 + ["new"] * 2
    return dict(Counter(words))


def main():
    freqs = toy_corpus_freqs()

    merges = learn_bpe_merges(freqs, num_merges=30)

    test_words = ["new", "newer", "lowest", "widest", "newestest"]

    print("\n=== Segmentations ===")
    for w in test_words:
        print(f"{w:>10} -> {apply_bpe(w, merges)}")


if __name__ == "__main__":
    main()


=== BPE Training ===
Initial vocab size (unique tokens): 11

Step 01: best_pair=('e', 'r') count=9 -> new_token='er' | vocab_size=11
Step 02: best_pair=('er', '_') count=9 -> new_token='er_' | vocab_size=11
Step 03: best_pair=('e', 'w') count=8 -> new_token='ew' | vocab_size=12
Step 04: best_pair=('n', 'ew') count=8 -> new_token='new' | vocab_size=11
Step 05: best_pair=('l', 'o') count=7 -> new_token='lo' | vocab_size=10
Step 06: best_pair=('lo', 'w') count=7 -> new_token='low' | vocab_size=10
Step 07: best_pair=('new', 'er_') count=6 -> new_token='newer_' | vocab_size=11
Step 08: best_pair=('low', '_') count=5 -> new_token='low_' | vocab_size=12
Step 09: best_pair=('d', 'er_') count=3 -> new_token='der_' | vocab_size=11
Step 10: best_pair=('i', 'der_') count=3 -> new_token='ider_' | vocab_size=10
Step 11: best_pair=('w', 'ider_') count=3 -> new_token='wider_' | vocab_size=9
Step 12: best_pair=('e', 's') count=2 -> new_token='es' | vocab_size=8
Step 13: best_pair=('es', 't') count=2 ->